# 2025 MCM Problem C · 编程演示

**45 分钟：从零基础到复现 O 奖论文的第一个模型**

---

### 这一节课要做什么

赛题给的是 1896–2024 共 30 届奥运会的奖牌、运动员、项目数据，要求预测 **2028 洛杉矶**的奖牌榜。
我们今天只做一件事：**把数据读进来 → 用 SVR 预测 2028 有多少个项目 → 画成图**。

这是 O 奖论文（上海交大 2514461 队）**Model I** 的完整复现。

### 怎么用这份 notebook

1. 确保数据文件夹 `2025_Problem_C_Data` 和这份 notebook **放在同一层目录**。
2. 菜单栏 `Run All`（或逐节按 `Shift+Enter`）。
3. 看到 `✏️ 先猜再跑` 的地方，**先自己猜一个数**，再运行看答案 —— 猜错才有印象。

### 目录

| 节 | 内容 | 时间 |
|---|---|---|
| 0 | 装库 + 环境自检（**建议课前先装好**） | 5 min |
| 1 | Python 语法最小集（6 个够了） | 5 min |
| 2 | 读数据：**pandas 入门** + **5 个数据坑** | 10 min |
| 3 | 建模：**SVR 预测 2028 项目数**（复现 O 奖 Model I） | 10 min |
| 4 | 分析 + 作图：项目数预测图、项目重要性热力图 | 10 min |
| 5 | 这个模型的**三个硬伤**（论文怎么写） | 4 min |

---
## 第 0 节 · 装库 + 环境自检（5 min）

### 0.1 装四个库

按你的系统，跑下面**两格中的一格**就行（装过的会秒过）。跑不通就换另一格试。

In [ ]:
# ===== Windows =====
# 提示 "pip 不是内部或外部命令" 的话，改成：python -m pip install ...
!pip install -q numpy pandas matplotlib scikit-learn

In [ ]:
# ===== Mac / Linux =====
# 提示权限不足的话，末尾加：--user
!pip3 install -q numpy pandas matplotlib scikit-learn

### 0.2 环境自检

**先跑这一格。** 如果这里报错，后面全白搭 —— 早发现早解决。

In [ ]:
import sys, importlib
print("Python 版本:", sys.version.split()[0])

ok = True
for mod, pip_name in [("numpy", "numpy"), ("pandas", "pandas"),
                      ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, "__version__", "?")
        print(f"  [OK]   {mod:12s} {v}")
    except ImportError:
        print(f"  [MISS] {mod:12s} 未安装 —— 回到 0.1 那一格重新装")
        ok = False

if not ok:
    print("\n还缺库。回到 0.1 那一格，或者去终端执行：")
    print("    Windows : pip install numpy pandas matplotlib scikit-learn")
    print("    Mac     : pip3 install numpy pandas matplotlib scikit-learn")
else:
    print("\n环境齐全，继续。")

# 一次性导入后面要用到的库（pandas 留到第 2 节正式登场时再导）
import numpy as np
import math, random, re
print("已导入: numpy / math / random / re")
print("numpy 版本:", np.__version__)

In [ ]:
from pathlib import Path

def find_data_dir():
    """自动找数据文件夹：先看当前目录和它的上几层，再看桌面"""
    roots = [Path.cwd()] + list(Path.cwd().parents)[:3]
    home = Path.home()
    roots += [home / "Desktop", home / "OneDrive" / "Desktop"]
    for r in roots:
        try:
            if not r.is_dir():
                continue
            direct = r / "2025_Problem_C_Data"
            if (direct / "summerOly_medal_counts.csv").exists():
                return direct.resolve()
            for p in r.glob("*/2025_Problem_C_Data"):
                if (p / "summerOly_medal_counts.csv").exists():
                    return p.resolve()
        except Exception:
            pass
    return None

DATA = find_data_dir()
if DATA is None:
    raise FileNotFoundError(
        "没找到 2025_Problem_C_Data 文件夹。\n"
        "请把这份 notebook 放到数据文件夹的同一层目录，然后重新运行本单元格。"
    )

print("数据目录:", DATA)
for f in sorted(DATA.glob("*.csv")):
    print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# 图里的所有标签统一用【英文】—— 不用管系统有没有中文字体，永远不会显示成方块
plt.rcParams["axes.unicode_minus"] = False          # 负号不变方块
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 110

print("matplotlib:", matplotlib.__version__)
print("图中标签: English")

---
## 第 1 节 · Python 语法最小集（5 min）

**不用学完 Python 再建模。** 做这一整节课的建模，你只需要下面 6 样东西：

变量 · 列表 · 字典 · for 循环 · if 判断 · 函数

下面全部用**真实的奥运数据**当例子（美国 2024 巴黎：40 金 44 银 42 铜）。

In [ ]:
# ① 变量 —— 就是给数据起名字
year = 2024                       # 整数 int
country = "美国"                   # 字符串 str
gold, silver, bronze = 40, 44, 42  # 一次给三个变量赋值
total = gold + silver + bronze

print(f"{year} 年 {country}: 金{gold} 银{silver} 铜{bronze}, 共 {total} 块")
print("类型分别是:", type(year).__name__, type(country).__name__, type(total).__name__)

In [ ]:
# ② 列表 list —— 一串有序的数
medals = [40, 44, 42]

print("第 1 个（金牌）:", medals[0])     # 注意：Python 从 0 开始数！
print("最后一个:", medals[-1])
print("求和:", sum(medals), "| 长度:", len(medals))

# 金3银2铜1 的加权得分 —— zip 把两个列表一对一对配起来
weights = [3, 2, 1]
score = sum(m * w for m, w in zip(medals, weights))
print("加权得分:", score)

In [ ]:
# ③ 字典 dict —— 用名字取值，比记下标方便
usa = {"year": 2024, "gold": 40, "silver": 44, "bronze": 42}

print("金牌数:", usa["gold"])

usa["total"] = usa["gold"] + usa["silver"] + usa["bronze"]   # 加一个新键
print("\n整个字典:")
for k, v in usa.items():
    print(f"   {k:8s} = {v}")

In [ ]:
# ④ for 循环 —— 对每个元素做同样的事
years    = [2008, 2012, 2016, 2020, 2024]
us_gold  = [  36,   48,   46,   39,   40]      # 美国近 5 届金牌数（从数据里查的）

for y, g in zip(years, us_gold):               # zip = 两个列表同步遍历
    bar = "#" * (g // 2)
    print(f"{y}: {g:2d} 金 {bar}")

print("\n5 届合计:", sum(us_gold), "金")

In [ ]:
# ⑤ if 判断 —— 分情况处理
g = 40
if g >= 40:
    tier = "第一梯队"
elif g >= 20:
    tier = "第二梯队"
else:
    tier = "第三梯队"
print(f"{g} 金 → {tier}")

# 顺带：列表推导式 = for + if 写在一行，建模里天天用
big_years = [y for y, g in zip(years, us_gold) if g >= 45]
print("金牌 45+ 的年份:", big_years)

In [ ]:
# ⑥ 函数 —— 把一段逻辑打包，反复用
def weighted_score(g, s, b, w=(3, 2, 1)):
    """金3银2铜1 的加权得分"""
    return g * w[0] + s * w[1] + b * w[2]

print("美国 2024:", weighted_score(40, 44, 42))
print("中国 2024:", weighted_score(40, 27, 24))

# 函数可以直接套在循环里 —— 这就是建模的雏形
print("\n美国近 5 届加权得分:")
for y, g, s, b in zip(years, [36,48,46,39,40], [39,26,37,41,44], [37,30,38,33,42]):
    print(f"   {y}: {weighted_score(g, s, b)}")

### ✅ 检查点 1

确认你能看懂这 6 段代码在干什么。看不懂没关系，往下走，第 2 节会反复用到它们 —— **用多了自然就懂**。

> 一句话总结：`list` 存一串，`dict` 按名字取，`for` 挨个处理，`if` 分情况，`函数` 打包复用。

---
## 第 2 节 · 用 pandas 读数据（10 min）

**pandas 是什么？** 把它理解成「Python 版的 Excel」就行：

- 一张表叫 **DataFrame**，有行、有列，可以按列名取值
- 读一个 csv 只要一行：`pd.read_csv(路径)`

建模比赛里 pandas 是默认选择 —— 读得快、筛得简单、能直接分组统计。
下面用真实数据过一遍它最常用的 5 个动作。

In [ ]:
# ---- 2.1 读进来：一行搞定 ----
import pandas as pd

mc = pd.read_csv(DATA / "summerOly_medal_counts.csv")
print("形状:", mc.shape, "（行 × 列）")
print("列名:", list(mc.columns))
mc.head()          # 单元格最后一行写变量名，会把表格直接显示出来

In [ ]:
# ---- 2.2 pandas 最常用的 5 个动作 ----

print("① 看前几行 / 形状")
print(mc.head(3).to_string())
print("形状:", mc.shape)

print("\n② 取一列（两种方式等价）")
print(mc["NOC"].head(3).to_string())

print("\n③ 按条件筛选 —— 美国 2024 拿了多少")
print(mc[(mc.Year == 2024) & (mc.NOC.str.strip() == "United States")]
      [["Gold", "Silver", "Bronze", "Total"]].to_string(index=False))

print("\n④ 分组统计 —— 每届一共发出多少块金牌")
print(mc.groupby("Year")["Gold"].sum().tail(5).to_string())

print("\n⑤ 整体概览 describe() —— 一眼看出量级和异常")
print(mc[["Gold", "Silver", "Bronze", "Total"]].describe().round(1).to_string())

### 记住这 6 行就够了

| 想干什么 | 代码 |
|---|---|
| 读 csv | `pd.read_csv(路径)` |
| 看前几行 / 看形状 | `df.head()` / `df.shape` |
| 取一列 | `df["列名"]` |
| 按条件筛 | `df[(df.Year == 2024) & (df.NOC == "China")]` |
| 分组求和 | `df.groupby("Year")["Gold"].sum()` |
| 统计概览 | `df.describe()` |

> ⚠️ 筛选条件里用的是 **`&`**，不是 `and`；每个条件都要单独加括号。
> 这是初学者最常踩的语法坑。

---

### ⚠️ 但是：pandas 读进来 ≠ 数据就干净了

下面 **5 个坑**，每一个都会让你的预测悄悄出错，而且**不报错**。
这是这节课最值钱的部分 —— 真实比赛里，清洗数据能占掉一半时间。

In [ ]:
# ===== 坑 1：NOC 列尾部藏着不可见字符 \xa0（不换行空格）=====
print("未清洗时国家数:", mc.NOC.nunique())

# 肉眼看起来一样，其实不一样：
bad = [s for s in mc.NOC.unique() if "\xa0" in str(s)][:3]
print("带 \\xa0 的例子（repr 才看得见）:", [repr(s) for s in bad])

clean_name = lambda s: str(s).replace("\xa0", " ").strip()
print("清洗后国家数:", mc.NOC.map(clean_name).nunique())
print("→ 凭空多出", mc.NOC.nunique() - mc.NOC.map(clean_name).nunique(), "个'假国家'")

In [ ]:
# 【图 1】清洗完，才画得出正确的图

# 👉 旋钮① CLEAN：改成 False 再跑一次，看美国那两个点去哪了
CLEAN = True
# CLEAN = False
# 👉 旋钮② COUNTRIES：换成 ["Japan", "South Korea"]、或 ["Germany", "East Germany", "West Germany"]
COUNTRIES = ["United States", "China"]
# COUNTRIES = ["Japan", "South Korea"]
COLORS = ["#c0392b", "#f39c12", "#2980b9", "#39a666"]

mc2 = mc.copy()
if CLEAN:
    mc2["NOC"] = mc2.NOC.map(clean_name)      # 去掉国家名尾部那个不可见字符

fig, ax = plt.subplots(figsize=(9.5, 3.9))
for name, color in zip(COUNTRIES, COLORS):
    d = mc2[mc2.NOC == name].groupby("Year")["Gold"].sum()
    ax.plot(d.index, d.values, marker="o", ms=3.5, lw=2, color=color, label=name)

ax.set_title("Fig1. Gold medals by Games" + (" (cleaned)" if CLEAN else " (RAW, not cleaned)"),
             fontsize=12.5)
ax.set_xlabel("Year"); ax.set_ylabel("Gold medals")
ax.legend(fontsize=10); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig1_medal_trend.png"); plt.show()

print("→ 中国 1984 才重返奥运；美国 1904 那个尖峰是圣路易斯奥运会（数据本身的老问题）。")
print("   看得出规律，也看得出异常 —— 这就是建模前必须先画图的原因。")
if CLEAN:
    print("\n👉 把 CLEAN 改成 False 重跑：美国 1932、1960 两届会凭空消失（1105 → 1027 金）。")
    print("   注意：**图上几乎看不出来** —— 线会直接跨过缺口，程序一个错都不报。")
    print("   这就是坑 1 最狠的地方：数错了还理直气壮。")
else:
    print("\n⚠️ 现在是【未清洗】状态：金牌总数只有 1027（真值 1105）。")
    print("   但你看图几乎发现不了 —— 这正是「不报错的数据错误」最危险的地方。")

In [ ]:
# ===== 坑 2：两张表的 NOC 列，根本不是一套体系 =====
ath = pd.read_csv(DATA / "summerOly_athletes.csv",
                  usecols=["Year", "NOC", "Team", "Sport", "Event", "Medal"])
print("运动员表:", ath.shape)

print("\n奖牌表  NOC 列的值（国名）:", mc.NOC.map(clean_name).head(3).tolist())
print("运动员表 NOC 列的值（三字码）:", ath.NOC.dropna().head(3).tolist())
print("运动员表 Team 列的值（国名）:", ath.Team.dropna().head(3).tolist())

# → 要联合两张表，必须先建 "三字码 → 国名" 的映射
code2name = (ath.dropna(subset=["NOC"])
                .groupby("NOC")["Team"]
                .agg(lambda s: s.value_counts().index[0]))
print("\n映射条数:", len(code2name), "| 例:", dict(list(code2name.items())[:5]))

In [ ]:
# ===== 坑 3：Team 列有 "Germany-1" / "Germany-2"（同一国多支队伍，比如沙排）=====
raw_teams = pd.read_csv(DATA / "summerOly_athletes.csv", usecols=["Team"], dtype=str).Team.dropna()
variants = sorted({t for t in raw_teams.unique() if t.startswith("Germany")})
print("Germany 的所有写法:", variants)

# 处理：去掉结尾的 "-数字"
import re
strip_suffix = lambda s: re.sub(r"-\d+$", "", s)
print("处理后:", sorted({strip_suffix(t) for t in variants}))

In [ ]:
# ===== 坑 4：同一块金牌，在两张表里行数不一样 =====
gold_rows_athletes = ((ath.Year == 2024) & (ath.Medal == "Gold")).sum()
gold_medals_table  = int(mc[mc.Year == 2024].Gold.sum())

print(f"2024 金牌：运动员表 {gold_rows_athletes} 行，奖牌表 {gold_medals_table} 块")
print(f"比值 {gold_rows_athletes/gold_medals_table:.2f}")
print("→ 团队项目一枚金牌，每个队员各占一行。想数奖牌就用奖牌表，想数人就用人表，别混。")

In [ ]:
# ===== 坑 5：programs.csv 有四麻烦 =====
prog = pd.read_csv(DATA / "summerOly_programs.csv", encoding="latin-1")   # ① 编码不是 utf-8
print("形状:", prog.shape)
print("列名:", list(prog.columns[:4]), "...", list(prog.columns[-3:]))

year_cols = [c for c in prog.columns if c[:4].isdigit()]
print(f"\n② 年份藏在【列名】里，共 {len(year_cols)} 届: {year_cols[0]} → {year_cols[-1]}")
print("   含 2028 吗?", "2028" in prog.columns, "  ← 不含！2028 项目数只能自己预测（O 奖模型 I 在干的事）")
print("   带星号的特殊列:", [c for c in prog.columns if "*" in c], "（1906 届间运动会，时间轴断点）")

# ③ 表里混着汇总行 —— 直接 sum 会翻倍算错
is_total = prog.Sport.astype(str).str.startswith("Total")
print("\n③ 混在数据里的汇总行:")
print(prog[is_total][["Sport", "Discipline", "2024"]].to_string(index=False))

ev = pd.to_numeric(prog["2024"], errors="coerce")
print(f"\n   直接 sum 所有行 : {ev.sum():.0f}   ← 错！汇总行也被算进去了")
print(f"   去掉汇总行之后  : {ev[~is_total].sum():.0f}   ← 2024 小项数（programs 表自己声明的）")
print(f"   奖牌表 2024 金牌: {int(mc[mc.Year == 2024].Gold.sum())}")
print("   → 差 1 块：一个小项发一块金牌，两边却对不上 —— 两张表的统计口径本来就不完全一致。")
print("   → 实战做法：选定一个口径（比如都用奖牌表的 328），并在论文里写清楚。")

In [ ]:
# 【图 2】每届到底有多少个小项？—— 它一直在变，所以 2028 必须预测

# 👉 旋钮③ DROP_TOTAL：改成 False 再跑一次，看整条曲线被抬高多少倍
DROP_TOTAL = True

ev_all = prog[year_cols].apply(pd.to_numeric, errors="coerce")
if DROP_TOTAL:
    ev_all = ev_all[~prog.Sport.astype(str).str.startswith("Total")]  # 剔除混进去的汇总行
tot_by_year = ev_all.sum().values
yrs = [int(str(c).rstrip("*")) for c in year_cols]                    # 1906* → 1906

fig, ax = plt.subplots(figsize=(9.5, 4.0))
ax.plot(yrs, tot_by_year, marker="o", ms=3.5, lw=1.8, color="#2980b9")
ax.scatter([2024], [tot_by_year[-1]], s=90, c="#c0392b", zorder=5)

# 三个关键年份的标注，错开摆，避免挤在一起
i0, i1, i2 = yrs.index(1896), yrs.index(2020), yrs.index(2024)
ax.annotate(f"1896: {int(tot_by_year[i0])}", xy=(1896, tot_by_year[i0]),
            xytext=(6, -16), textcoords="offset points", fontsize=9.5)
ax.annotate(f"2020: {int(tot_by_year[i1])}", xy=(2020, tot_by_year[i1]),
            xytext=(-34, 10), textcoords="offset points", fontsize=9.5)
ax.annotate(f"2024: {int(tot_by_year[i2])}", xy=(2024, tot_by_year[i2]),
            xytext=(-16, -46), textcoords="offset points", fontsize=9.5, color="#c0392b",
            ha="right", bbox=dict(fc="white", ec="none", alpha=.85, pad=1.5))

ax.set_xlim(1888, 2032); ax.set_ylim(0, max(tot_by_year) * 1.18)
ax.annotate("2028 = ?\n(not in data)",
            xy=(2028, max(tot_by_year) * 0.55), fontsize=10.5,
            color="#c0392b", ha="center")

ax.set_title("Fig2. Total events by Games: not monotonic, 2028 must be predicted"
             + (" (summary rows removed)" if DROP_TOTAL else " (RAW, summary rows included)"),
             fontsize=12.5)
ax.set_xlabel("Year"); ax.set_ylabel("Number of events")
ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig2_events_by_year.png"); plt.show()

print("注意 2020(339) → 2024(329) 反而【降了】—— 项目数不是只增不减。")
print("所以 2028 有多少项，只能靠模型外推（第 3 节就干这个）。")
if DROP_TOTAL:
    print("\n👉 把 DROP_TOTAL 改成 False 重跑：2024 会从 329 变成 738（虚高 2.24 倍）")
    print("   但曲线【形状几乎不变】—— 看着挺像，其实每一届都是错的。")
else:
    print("\n⚠️ 现在是【含汇总行】状态：738 = 真值 329 + 三个 Total 行被重复加了一遍。")

### ✅ 检查点 2

**五个坑**，每一个都不报错、但会悄悄毁掉你的预测：

| # | 坑 | 后果 |
|---|---|---|
| 1 | NOC 尾部不可见字符 `\xa0` | 多出 46 个"假国家"，预测被摊薄 |
| 2 | 两张表 NOC 是两套体系（国名 vs 三字码） | 联合分析直接对不上 |
| 3 | Team 有 `Germany-1/2/3` 后缀 | 同一国被拆成多支 |
| 4 | 团队项目一枚金牌在人表里占多行 | 743 行 vs 328 块，数错奖牌 |
| 5 | programs 编码 latin-1、年份在列名、混了汇总行、**没有 2028** | 直接 sum 得 738（真值 329） |

最后一件事：把上面零散的清洗动作**打包成一个函数**，后面一直用它。


In [ ]:
def clean(s):
    """一条龙清洗：去不可见空格 → 去队名后缀 → 去首尾空白"""
    return re.sub(r"-\d+$", "", str(s).replace("\xa0", " ")).strip()

mc_clean = mc.copy()
mc_clean["NOC"] = mc_clean.NOC.map(clean)
print("清洗后:", mc_clean.shape, "| 国家数:", mc_clean.NOC.nunique(),
      "| 届数:", mc_clean.Year.nunique(), f"({mc_clean.Year.min()}–{mc_clean.Year.max()})")
mc_clean.head()

---
## 第 3 节 · 建模：SVR 预测 2028 项目数（12 min）

这一节复现 O 奖论文的 **Model I**。

**为什么第一步是预测项目数？**

因为一个小项发一块金牌 —— **金牌总数 = 项目总数**。2028 有多少项目不知道，
奖牌榜的总量就没法定下来（这就是赛题里的"守恒约束"）。
而 `programs.csv` 的最后一列是 **2024**，压根没有 2028。

**一句话讲清思路**

> 每个项目（比如艺术体操）在历届奥运会上有多少个小项，本身就是一条**随时间变化的曲线**。
> 把这条曲线拟合出来，往后延到 2028，就是预测值。

剩下的问题只有一个：**怎么拟合这条弯弯曲曲的曲线** —— 用 SVR。

In [ ]:
# ---- 3.1 先把时间序列抽出来 ----
# programs.csv 的年份藏在列名里（第 2 节坑 5），还得先剔除混进去的汇总行
YEAR_COLS = [c for c in prog.columns if c[:4].isdigit()]
YEARS = np.array([int(str(c).rstrip("*")) for c in YEAR_COLS], float)   # 1906 是届间运动会

is_total = prog.Sport.astype(str).str.startswith("Total")
DIS = prog[~is_total].reset_index(drop=True)
print("分项数:", len(DIS), "| 届数:", len(YEAR_COLS), "| 年份:", int(YEARS.min()), "→", int(YEARS.max()))

def series_of(code):
    """取某个分项的历史项目数序列。NaN = 那届没设这个项目，按 0 处理"""
    v = pd.to_numeric(DIS.loc[DIS.Code == code, YEAR_COLS].iloc[0], errors="coerce").values
    return np.nan_to_num(v)

# GAR = Artistic Gymnastics（艺术体操）—— O 奖论文就是拿它当例子的
gar = series_of("GAR")
print("\n艺术体操 GAR 历年项目数：")
print("  " + "  ".join(f"{int(y)}:{int(n)}" for y, n in zip(YEARS, gar)))

### ✏️ 先猜再跑

艺术体操 2024 是 **14 项**，而且从 1952 年起就没怎么变过（上面那串数字的后半段几乎全是 14）。

**你觉得模型会预测 2028 是多少项？** 先记住你的答案，再跑下一格。

### 3.2 SVR 是什么？（三句话说完）

1. **普通线性回归**只会画一条直线。但项目数的变化是弯的（早期暴涨 → 中期平台 → 近期微调）。
2. **SVR（支持向量回归）**能画一条平滑的曲线去贴合这些点，还能容忍少量离群点不被带跑。
   O 奖选它而不用 ARIMA，就是因为这个"弯"字。
3. SVR 有几个参数（`C`、`gamma`、`epsilon`）决定曲线画得多弯、多贴合。
   **不用手调** —— 用**交叉验证**让程序自己挑最好的一组。

> 大一阶段你只要记住：**SVR = 能拟合非线性曲线的回归**，参数交给 `GridSearchCV`。

**流程**（对应 O 奖 Fig 1）：

```
拿出一条历史序列 → 8:2 切训练/测试 → 标准化
                → 交叉验证挑参数 → 用最好的模型外推 2028
```

In [ ]:
# ---- 3.3 对一个分项建 SVR 模型 ----
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

def svr_forecast(vals, random_state=42):
    """输入一条历史序列，返回 (2028预测值, MSE, R平方, 最优参数, 拟合好的模型)"""
    X = YEARS.reshape(-1, 1)                 # 自变量只有一个：年份
    y = vals.astype(float)

    # ① 8:2 切训练/测试（留出一部分数据，好在旁边检验模型准不准）
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=random_state)

    # ② 标准化：年份是 1896~2024，项目数是 0~50，量纲差太远，不标准化会出问题
    sx = StandardScaler().fit(X_tr)
    sy = StandardScaler().fit(y_tr.reshape(-1, 1))

    # ③ 交叉验证自动挑参数
    grid = {"C": [1, 10, 100, 1000],
            "gamma": ["scale", 0.01, 0.1, 1],
            "epsilon": [0.01, 0.1, 0.5]}
    gs = GridSearchCV(SVR(kernel="rbf"), grid, cv=5, scoring="neg_mean_squared_error")
    gs.fit(sx.transform(X_tr), sy.transform(y_tr.reshape(-1, 1)).ravel())

    # ④ 在测试集上打分（记得把标准化还原回去，否则算的是标准化尺度上的误差）
    pred_te = sy.inverse_transform(gs.predict(sx.transform(X_te)).reshape(-1, 1)).ravel()
    mse = mean_squared_error(y_te, pred_te)
    r2  = r2_score(y_te, pred_te)

    # ⑤ 外推 2028
    p2028 = float(sy.inverse_transform(gs.predict(sx.transform([[2028.0]])).reshape(-1, 1))[0, 0])
    return p2028, mse, r2, gs.best_params_, (gs.best_estimator_, sx, sy)

p_gar, mse_gar, r2_gar, best_gar, fitted_gar = svr_forecast(gar)

print("艺术体操 GAR → 2028 预测:", round(p_gar, 2), "项")
print("   （O 奖论文给的也是 14 项 —— 复现对了 ✓）")
print()
print("测试集 MSE:", round(mse_gar, 3), "  R平方:", round(r2_gar, 3))
print("   MSE  = 预测值和真实值的平均平方误差 → 越小越好")
print("   R平方 = 模型解释了多少波动 → 越接近 1 越好（0.57 算中等偏上）")

### 3.4 71 个分项全跑一遍，汇总出 2028 项目总数

（下面这格要跑 **约 20 秒**：71 个模型 × 48 组参数 × 5 折交叉验证。可以先讲再等。）

In [ ]:
rows = []
for _, r in DIS.iterrows():
    v = pd.to_numeric(r[YEAR_COLS], errors="coerce").values
    if np.all(np.isnan(v)):
        continue
    v = np.nan_to_num(v)
    p28, m, r2v, bp, _ = svr_forecast(v)
    rows.append({"Sport": r.Sport, "Discipline": r.Discipline, "Code": r.Code,
                 "Y2024": int(v[-1]), "Y2028": float(np.clip(p28, 0, None)),
                 "MSE": m, "R2": r2v})

PRED = pd.DataFrame(rows)
PRED["Y2028"] = PRED.Y2028.round().astype(int)

print("建模分项数:", len(PRED))
print("2024 实际项目总数:", int(PRED.Y2024.sum()))
print("2028 预测项目总数:", int(PRED.Y2028.sum()))

In [ ]:
# 【图 3】哪些大项会变？把 2024 实际 和 2028 预测 并排画出来
by_sport = PRED.groupby("Sport")[["Y2024", "Y2028"]].sum()
by_sport["变化"] = by_sport.Y2028 - by_sport.Y2024
top = by_sport.reindex(by_sport["变化"].abs().sort_values(ascending=False).index).head(8)

ypos = np.arange(len(top))
fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.barh(ypos + 0.19, top.Y2024, height=.38, color="#95a5a6", label="2024 actual")
ax.barh(ypos - 0.19, top.Y2028, height=.38, color="#c0392b", label="2028 predicted")
ax.set_yticks(ypos)
ax.set_yticklabels(list(top.index), fontsize=10)
ax.invert_yaxis()                                    # 变化最大的排最上面
ax.set_title("Fig3. Biggest movers: 2024 actual vs 2028 predicted", fontsize=12.5)
ax.set_xlabel("Number of events")
ax.legend(fontsize=10, loc="lower right"); ax.grid(alpha=.3, axis="x")
plt.tight_layout(); plt.savefig("fig3_change_2028.png"); plt.show()

print("灰=2024 有几个小项，红=模型预测 2028 有几个。")
print("滑板、攀岩的红色柱子直接掉到 0 —— 为什么，第 5 节会讲。")

### ✅ 检查点 3

- 用 **SVR** 把 71 个分项的历史序列外推到 2028，得到 **≈ 316 项**（2024 是 329）
- 艺术体操 **14 项**，与 O 奖完全一致
- 总数系统性偏低：**近期新增项目全被预测成 0**，原因在第 5 节展开

---
## 第 4 节 · 分析 + 作图（10 min）

两件事：

1. **先把第 3 节的结果画出来** —— 项目数预测图（对应 O 奖 Fig 2）
2. **再做一个分析，把它也画出来** —— 哪些项目对哪些国家最重要（对应 O 奖 Fig 3）

（图里的标题、坐标轴、图例统一用**英文**：既省掉中文字体的麻烦，也方便直接插进英文论文。
讲解文字仍然是中文。）

In [ ]:
# 【图 4】艺术体操项目数 —— 历史 + SVR 拟合 + 2028 外推（对应 O 奖 Fig 2）
model, sx, sy = fitted_gar
gx = np.linspace(YEARS.min(), 2028, 400).reshape(-1, 1)
gy = sy.inverse_transform(model.predict(sx.transform(gx)).reshape(-1, 1)).ravel()

fig, ax = plt.subplots(figsize=(9.5, 4.3))
ax.scatter(YEARS, gar, s=26, c="#2980b9", alpha=.75, zorder=3, label="Historical")
ax.plot(gx.ravel(), gy, color="#c0392b", lw=2, zorder=2, label="SVR fit")
ax.scatter([2028], [p_gar], s=170, marker="*", c="#f39c12", edgecolor="black", zorder=5,
           label=f"2028 forecast {p_gar:.1f}")
ax.axvline(2024, color="#7f8c8d", ls=":", lw=1.2)
ax.annotate("history  |  extrapolation ->",
            xy=(2022.5, max(gar) * 0.28), fontsize=9, color="#7f8c8d", ha="right")

ax.set_title(f"Fig4. Artistic gymnastics: history and 2028 forecast "
             f"(MSE={mse_gar:.2f}, R2={r2_gar:.2f})", fontsize=12.5)
ax.set_xlabel("Year")
ax.set_ylabel("Number of events")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig4_svr_events.png"); plt.show()

### 4.1 项目重要性：加权得分 S（Model I 4.4）

换个角度：不预测项目数了，回头看 **2024 年各国是靠哪些项目拿分的**。
O 奖用加权得分衡量"哪个项目对这个国家最重要"：

$$S = 3 \times N_{金} \;+\; 2 \times N_{银} \;+\; 1 \times N_{铜}$$

某项目占一国总分的比例越高 → 这个项目对该国越重要。**下一张图就画这个。**

In [ ]:
# ---- 2024 各国 × 各项目 的加权得分 ----
m24 = ath[(ath.Year == 2024) & (ath.Medal != "No medal")].copy()
m24["Team"] = m24.Team.map(lambda s: re.sub(r"-\d+$", "", str(s)).strip())

# 团队项目一枚奖牌多人各占一行（坑 4），必须按项目去重
uniq = m24.drop_duplicates(["Team", "Sport", "Event", "Medal"]).copy()
print("2024 有奖牌的记录:", len(m24), "行 → 按项目去重后:", len(uniq), "块奖牌")

# 👉 旋钮④ W（权重）：(1,0,0) 只看金牌 / (1,1,1) 数奖牌 / (5,3,1) 更重金牌 —— 排名会换人
W = (3, 2, 1)

uniq["w"] = uniq.Medal.map({"Gold": W[0], "Silver": W[1], "Bronze": W[2]})
score = uniq.groupby(["Team", "Sport"])["w"].sum().reset_index()
total = score.groupby("Team")["w"].sum()
score["占比"] = score.w / score.Team.map(total)

print("\n2024 总得分前 6 名:")
print("  " + "  ".join(f"{k}:{v}" for k, v in total.nlargest(6).items()))

print("\n美国得分最高的 5 个项目:")
print(score[score.Team == "United States"].nlargest(5, "w")[["Sport", "w", "占比"]].to_string(index=False))
print("\n注: 这里是【运动员表】口径（按项目去重后算的），")
print("    和第 1 节用【奖牌表】算出的分数会有出入 —— 两张表统计口径不同，不是算错了。")

In [ ]:
# 【图 5】Top 10 国家的加权得分，拆成金 / 银 / 铜三段
cnt = (uniq.groupby(["Team", "Medal"]).size()
           .unstack(fill_value=0)
           .reindex(columns=["Gold", "Silver", "Bronze"], fill_value=0))
part = pd.DataFrame({"gold": cnt["Gold"] * W[0],       # ← 跟着上面的 W 走，别写成 3/2/1
                     "silver": cnt["Silver"] * W[1],
                     "bronze": cnt["Bronze"] * W[2]})
part["S"] = part.sum(axis=1)
top10 = part.nlargest(10, "S")

fig, ax = plt.subplots(figsize=(9.5, 4.6))
left = np.zeros(len(top10))
for (col, color, name), w in zip([("gold", "#f1c40f", "Gold"),
                                  ("silver", "#bdc3c7", "Silver"),
                                  ("bronze", "#cd7f32", "Bronze")], W):
    if w == 0:                                  # 权重为 0 就整段不画（图更干净）
        continue
    ax.barh(range(len(top10)), top10[col], left=left, color=color,
            label=f"{name} x{w}", edgecolor="white", lw=.5)
    left = left + top10[col].values

ax.set_yticks(range(len(top10)))
ax.set_yticklabels(list(top10.index), fontsize=10)
ax.invert_yaxis()
pad = top10.S.max() * 0.02          # 标签离条端的距离按比例给，换权重也好看
for i, v in enumerate(top10.S):
    ax.text(v + pad, i, str(int(v)), va="center", fontsize=9.5)
ax.set_xlim(0, top10.S.max() * 1.14)
ax.set_title(f"Fig5. Weighted score S = {W[0]}*gold + {W[1]}*silver + {W[2]}*bronze, 2024 (Top 10)",
             fontsize=12.5)
ax.set_xlabel("Weighted score S")
ax.legend(fontsize=10, loc="lower right"); ax.grid(alpha=.3, axis="x")
plt.tight_layout(); plt.savefig("fig5_country_score.png"); plt.show()

print(f"权重 W={W} 时：{top10.index[0]} {int(top10.S.iloc[0])} 分、"
      f"{top10.index[1]} {int(top10.S.iloc[1])} 分 —— 但「怎么拿到的」完全不同，看下一张图。")
print("👉 把 W 改成 (1,0,0) 再跑：只看金牌，冠军会换成中国（本表口径 40 vs 39 金）。")
print("   指标怎么定义，决定结论是什么 —— 写论文必须做这种敏感性分析。")

In [ ]:
# 【图 6】把上面的占比画成热力图（对应 O 奖 Fig 3）
# 👉 旋钮⑤ CMAP：换成 "Blues" / "viridis" / "coolwarm" 再跑，配色差别非常大
CMAP = "YlOrRd"

top6 = total.nlargest(6).index.tolist()
piv = (score[score.Team.isin(top6)]
       .pivot(index="Team", columns="Sport", values="占比")
       .fillna(0).loc[top6])
keep = piv.sum().nlargest(10).index          # 只留最重要的 10 个项目
piv = piv[keep] * 100

fig, ax = plt.subplots(figsize=(10.5, 4.4))
im = ax.imshow(piv.values, cmap=CMAP, aspect="auto", vmin=0)
ax.set_xticks(range(len(keep)))
ax.set_xticklabels(list(keep), rotation=30, ha="right", fontsize=9.5)
ax.set_yticks(range(len(top6)))
ax.set_yticklabels(list(top6), fontsize=10)

cmap_obj = plt.get_cmap(CMAP)                 # 换配色时，格子里的字要跟着换黑白
vmax = piv.values.max()
for i in range(piv.shape[0]):                 # 格子里写上数值
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if v >= 1:
            r, g, b, _ = cmap_obj(v / vmax)              # 这一格的底色
            dark_text = (0.299 * r + 0.587 * g + 0.114 * b) > 0.55   # 底色亮 → 写黑字
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8.5,
                    color="black" if dark_text else "white")

ax.set_title("Fig6. Share of each country's weighted score by sport, 2024 (%)",
             fontsize=12.5)
plt.colorbar(im, ax=ax, shrink=.85, label="share (%)")
plt.tight_layout(); plt.savefig("fig6_sport_share.png"); plt.show()

print("O 奖原文的结论：美国靠田径+游泳、日本靠摔跤、中国靠水上项目 —— 对照上图，对得上。")

### ✅ 检查点 4

三张图的分工：

1. **图 4 项目数预测图** —— 主结果图。散点是历史、曲线是模型、星点是外推；
   **把 MSE 和 R² 写进标题**，评审一眼就能看到你做了误差评估
2. **图 5 得分堆叠图** —— 谁强、强多少，一眼可比（美国 242 / 中国 198）
3. **图 6 重要性热力图** —— 体现"深入分析"。美国田径 30% + 游泳 23%，
   日本摔跤独占 29% —— 这种国家差异，后面预测奖牌时是有用的先验

**这节课一共 6 张图**（`fig1` ~ `fig6`），都存在 notebook 同目录下，可直接插进 LaTeX 论文。

### 🎛️ 这一节的两个旋钮

| 旋钮 | 改成 | 会看到 |
|---|---|---|
| `W`（图 5 上面那格） | `(1,0,0)` 只看金牌 | **冠军换人**：中国 40 > 美国 39（本表口径） |
| `W` | `(1,1,1)` 数奖牌 / `(5,3,1)` | 美国稳第一，但 `(5,3,1)` 下**澳 160 反超英 159** |
| `CMAP`（图 6 那格） | `"Blues"` / `"viridis"` | 配色差异极大；`viridis` 对色盲友好 |

> 改完 `W` 要**连着图 5、图 6 两格一起重跑**（图 6 的占比也用同一套权重）。

---
## 第 5 节 · 这个模型的三个硬伤（4 min）

前三节跑通了一个能用的模型。但**能把模型跑通 ≠ 会写论文** ——
真正拉开分差的，是你能不能讲清楚这个模型**在哪里不可信**。

下面三点，每一点都能直接写进论文的「模型局限性」一节。

> 参考：O 奖论文 **2514461 队《Sketch an Olympic Vision》（上海交通大学）**，
> 我们复现的是它的 Model I。

### 硬伤 ①：它预测不出"从无到有"的新项目

2028 洛杉矶要加板球、棍网球、腰旗橄榄球、壁球。这些在 `programs.csv` 里都有行，
但历史全是 0，SVR 只能给 0；更离谱的是，图 3 里那些 2024 明明有项目的大项，
也有 **6 个被预测成了 0**（3x3 篮球、霹雳舞、BMX 自由式、滑板、攀岩、冲浪）
—— 因为它们的序列在 31 届里几乎全是 0，SVR 只能拟合出"约等于 0"。

后果是**项目总数系统性偏低**（316 < 329）。而奖牌榜预测是建立在项目数之上的，
**这个误差会一路传到下游**。

> 写进论文的一句话：本方法无法刻画项目设置的**结构性变化**（新增 / 取消项目）。

### 硬伤 ②：只给了一个数，没有区间

SVR 给的是点估计（艺术体操 14 项），**没说这个 14 有多可靠**。
论文里应该配上**敏感性分析**：换个参数网格、换个切分方式再跑一遍，看结论稳不稳。
给一个区间（比如 12~16 项），比给一个孤零零的 14 可信得多。

### 硬伤 ③：用随机切分评估时间序列，本身就不严谨

我们（和 O 奖一样）是 **8:2 随机**抽 20% 当测试集。但时间序列的正确做法
是**按时间切**（用早年预测晚年），否则等于"用未来的数据预测过去"，会高估模型能力。

不过真按时间切也有麻烦：近 30 年恒定不变的序列（比如体操一直是 14）会让 R² 失去意义。
**这是一个明知不完美、也只能如此的妥协** —— 写论文时如实说明即可，
不要假装自己是按时间切的。

---

**那可以往哪改？（只给方向，不展开）**

- 只对近 20~30 年建模，别让 1900 年的 0 拖累 2028
- 对"近两届有值"的项目单独处理 —— 这是修掉 316 最直接的办法
- 做**情景分析**：保守 / 基准 / 乐观三档，给区间而不是一个孤零零的数
- 引入数据之外的信息（2028 的项目设置是公开的）

---
## 收尾

### 今天你实际跑通了什么

- 用 `pandas` 读了表，清洗掉 **46 个假国家**
- 用 **SVR + 高斯核** 复现了 O 奖 Model I，71 个分项外推到 2028
- 艺术体操预测 **14 项**，与 O 奖完全一致
- 发现并讲清了 Model I 的硬伤：**新项目一律预测为 0，总数系统性偏低**
- 画了 **6 张可以直接插进论文的图**（`fig1` ~ `fig6`，按讲课顺序编号）

### 建模的通用套路（换个赛题也能用）

```
读数据 → 清洗（占一半时间）→ 定指标 → 建模型 → 常识检验 → 找模型的边界 → 画图
```

**「常识检验」和「找边界」这两步最容易被跳过，也最能拉开分差。**

### 交作业前自查

- [ ] 每个数都有出处（哪个表、怎么算的）
- [ ] 报告了模型的**误差指标**（MSE / R²），不是只给一个预测数
- [ ] 做过参数敏感性分析
- [ ] 图和表都有标题、有单位
- [ ] 诚实写了模型的局限（比如新项目预测为 0）